In [ ]:
import os
import pandas as pd
import re

def extract_dscd(x):
    if not isinstance(x, str):
        return None
    m = re.match(r"(?:DPL#\()?([^(]+)\(", x)
    return m.group(1) if m else None

variable_names = [
    #"WC03501",
    "WC02999",
    "WC02003",
    "WC01001",
    "EPS",
    "DY",
    "PE",
    "WC08301",
    "WC08326",
    "WC03255",
    "WC03251",
    "WC03051",
    "WC08231",
    "WC18198",
    "WC04601",
    "ENERO52V",
    "ENERO55V",
    "ENERO132V"
]

failed_files = []

for variable_name in variable_names:
    #variable_name = "EPSISURSUE"
    folder = f"D:/Datastream/Firmcharacteristics_Monthly/US/"

    filenames = sorted(f for f in os.listdir(os.path.join(folder, variable_name)) if f.endswith('.xlsx'))
    df_files = []
    for file in filenames:
        try:
            df_tmp = pd.read_excel(os.path.join(folder, variable_name, file), sheet_name="Sheet", engine='openpyxl')
            df_tmp = df_tmp.loc[:, ~df_tmp.columns.str.startswith('#ERROR')]

            #extract DSCD codes
            clean_ids = [extract_dscd(x) for x in df_tmp.iloc[0, 1:].tolist()]
            df_tmp = df_tmp.iloc[2:].copy()
            df_tmp.rename(columns={df_tmp.columns[0]: "Date"}, inplace=True)
            df_tmp["Date"] = pd.to_datetime(df_tmp["Date"])
            df_tmp.loc[:, df_tmp.columns != "Date"] = df_tmp.loc[:, df_tmp.columns != "Date"].apply(pd.to_numeric, errors="coerce")

            df_tmp.columns = ["Date"] + clean_ids
            df_tmp = df_tmp.melt(id_vars = "Date", var_name = "DSCD", value_name = variable_name)
            df_files.append(df_tmp)
            print(f"Finished import of file {file}")
        except Exception as e:
            # Do not silently skip a batch (~1000 firms): record the failure and report it below.
            print(f"An error occurred for file {file}: {type(e).__name__}: {e}")
            failed_files.append({"variable": variable_name, "file": file, "error": f"{type(e).__name__}: {e}"})

    df = pd.concat(df_files)
    df = df.dropna(subset=variable_name)
    df.to_csv(os.path.join(folder, "Paneldata", f"{variable_name}.csv"), index = False)

# Report failed files. A failed file means that one list (~1000 firms) is missing for this variable.
if failed_files:
    failed_df = pd.DataFrame(failed_files)
    failed_df.to_csv(os.path.join(folder, "Paneldata_failed_files.csv"), index=False)
    raise RuntimeError(f"{len(failed_files)} file(s) could not be imported, see Paneldata_failed_files.csv:\n{failed_df}")
